# 03 · Dependencia espacial: Moran, Geary, LISA y Getis-Ord Gi*

**Pregunta:** ¿la pérdida de vegetación ocurre al azar o forma conglomerados?

Matriz de pesos **W** a partir de la vecindad H3 de primer orden (k-ring = 1), estandarizada por filas:

$$w_{ij}^{*} = \frac{w_{ij}}{\sum_j w_{ij}}, \qquad w_{ij} = 1 \text{ si } j \in \text{kring}(i,1),\; j \ne i$$

Además se calcula Gi* **dentro de Spark** con un *self-join* sobre `h3_kring`: el mismo
estadístico escala a millones de hexágonos sin construir W en memoria.

In [ ]:
%pip install -q rasterio pyproj h3 geopandas shapely esda libpysal spreg pykrige scikit-gstat mapclassify
dbutils.library.restartPython()

In [ ]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# Raíz del repositorio (Git folder en Databricks o clon local)
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from geocloud import config as C, dbx
spark = dbx.get_spark()
VOL = dbx.volume_path()
FIG = Path(VOL) / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("Databricks:", dbx.is_databricks(), "| Volume:", VOL)

In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from geocloud import spatial

feat = spark.table(dbx.tbl("gold_h3_features")).toPandas()
YEARS = sorted(int(c.split("_")[1]) for c in feat.columns if c.startswith("ndvi_") and c.split("_")[1].isdigit())
Y0, Y1 = YEARS[0], YEARS[-1]
gdf = spatial.h3_to_geodataframe(feat, crs_out=C.CRS_UTM)
print(f"Hexágonos: {len(gdf):,} | años: {YEARS}")

In [ ]:
from esda.moran import Moran, Moran_Local
from esda.geary import Geary
from esda.getisord import G_Local
from libpysal.weights import lag_spatial

d = gdf.dropna(subset=["d_ndvi"]).reset_index(drop=True)
w = spatial.h3_weights(d.h3, k=1)
if w.islands:
    d = d[~d.h3.isin(w.islands)].reset_index(drop=True)
    w = spatial.h3_weights(d.h3, k=1)
print(f"n = {w.n:,} | vecinos medios = {w.mean_neighbors:.2f} | islas = {len(w.islands)}")

## 1. Autocorrelación global

In [ ]:
rows = []
for var in [v for v in ["d_ndvi", "d_pert", "frac_perdida_oficial", "frac_perdida_hansen"] if v in d]:
    y = d[var].values
    mi, gc = Moran(y, w, permutations=999), Geary(y, w, permutations=999)
    rows.append({"variable": var, "I de Moran": mi.I, "E[I]": mi.EI, "z(I)": mi.z_sim, "p(I)": mi.p_sim,
                 "C de Geary": gc.C, "p(C)": gc.p_sim})
glob = pd.DataFrame(rows).round(4); glob

**Lectura:** I > 0 y C < 1 con p < 0.05 indican autocorrelación positiva: hexágonos con pérdida
están rodeados de hexágonos con pérdida. La ley de Tobler se cumple con fuerza en la minería aluvial.

In [ ]:
# Evolución temporal del I de Moran del NDVI (dependencia espaciotemporal)
evo = []
for yv in YEARS:
    sub = gdf.dropna(subset=[f"ndvi_{yv}"]).reset_index(drop=True)
    wy = spatial.h3_weights(sub.h3, k=1)
    keep = ~sub.h3.isin(wy.islands)
    sub = sub[keep].reset_index(drop=True); wy = spatial.h3_weights(sub.h3, k=1)
    evo.append({"anio": yv, "I_moran_ndvi": Moran(sub[f"ndvi_{yv}"].values, wy, permutations=199).I, "n": wy.n})
evo = pd.DataFrame(evo); print(evo.round(4))

In [ ]:
y = d["d_ndvi"].values
z = (y - y.mean()) / y.std()
lag = lag_spatial(w, z)
mi = Moran(y, w, permutations=999)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(z, lag, s=3, alpha=0.3)
b = np.polyfit(z, lag, 1); xx = np.linspace(z.min(), z.max(), 10)
ax.plot(xx, b[0] * xx + b[1], c="r", label=f"pendiente = I = {mi.I:.3f}")
ax.axhline(0, c="k", lw=0.6); ax.axvline(0, c="k", lw=0.6)
ax.set_xlabel("ΔNDVI estandarizado"); ax.set_ylabel("Rezago espacial W·z"); ax.legend()
ax.set_title("Diagrama de dispersión de Moran")
plt.tight_layout(); plt.savefig(FIG / "03_moran_scatter.png", dpi=150); plt.show()

## 2. LISA: conglomerados locales (Anselin, 1995)

In [ ]:
lisa = Moran_Local(y, w, permutations=999, seed=42)
labels = {1: "Alto-Alto", 2: "Bajo-Alto", 3: "Bajo-Bajo", 4: "Alto-Bajo"}
d["lisa"] = np.where(lisa.p_sim < 0.05, pd.Series(lisa.q).map(labels), "No significativo")
colors = {"Alto-Alto": "#d7191c", "Bajo-Bajo": "#2c7bb6", "Bajo-Alto": "#abd9e9",
          "Alto-Bajo": "#fdae61", "No significativo": "#eeeeee"}
fig, ax = plt.subplots(figsize=(12, 7))
for k, c in colors.items():
    d[d.lisa == k].plot(ax=ax, color=c, linewidth=0, label=k)
ax.legend(loc="lower left", title="ΔNDVI (p < 0.05)")
ax.set_title(f"Clusters LISA del cambio de NDVI {Y0}-{Y1}"); ax.set_axis_off()
plt.tight_layout(); plt.savefig(FIG / "03_lisa.png", dpi=150); plt.show()
print(d.lisa.value_counts())

**Lectura:** los clusters **Bajo-Bajo** (ΔNDVI muy negativo rodeado de lo mismo) son los frentes
de deforestación minera. Los Alto-Alto son zonas de recuperación o bosque estable.

## 3. Getis-Ord Gi*: puntos calientes de perturbación

Gi* requiere una variable no negativa; se usa la fracción de píxeles perturbados del último año.

In [ ]:
var = f"pert_{Y1}"
dg = gdf.dropna(subset=[var]).reset_index(drop=True)
wb = spatial.h3_weights(dg.h3, k=1, row_standardize=False)
keep = ~dg.h3.isin(wb.islands); dg = dg[keep].reset_index(drop=True)
wb = spatial.h3_weights(dg.h3, k=1, row_standardize=False)
gi = G_Local(dg[var].values, wb, transform="B", star=True, permutations=0)
dg["gi_z"] = gi.Zs
dg["hotspot"] = pd.cut(dg.gi_z, [-np.inf, -2.58, -1.96, 1.96, 2.58, np.inf],
                       labels=["Frío 99%", "Frío 95%", "No sig.", "Caliente 95%", "Caliente 99%"])
fig, ax = plt.subplots(figsize=(12, 7))
dg.plot(column="gi_z", cmap="coolwarm", vmin=-6, vmax=6, ax=ax, linewidth=0, legend=True,
        legend_kwds={"shrink": 0.6, "label": "Gi* (z)"})
ax.set_title(f"Puntos calientes Getis-Ord Gi* de perturbación {Y1}"); ax.set_axis_off()
plt.tight_layout(); plt.savefig(FIG / "03_gi_star.png", dpi=150); plt.show()
print(dg.hotspot.value_counts())

## 4. Gi* escalable dentro de Spark SQL

$$G_i^* = \frac{\sum_j w_{ij}x_j - \bar{X}\sum_j w_{ij}}{S\sqrt{\dfrac{n\sum_j w_{ij}^2 - (\sum_j w_{ij})^2}{n-1}}}$$

Con pesos binarios que incluyen al propio hexágono, $\sum_j w_{ij}^2 = \sum_j w_{ij}$ = número de
vecinos presentes + 1. Solo se necesitan un `explode(h3_kring(...))` y un `JOIN`.

In [ ]:
sdf = spark.createDataFrame(dg[["h3", var]].rename(columns={var: "x"}))
sdf.createOrReplaceTempView("base")
gi_spark = spark.sql("""
WITH stats AS (SELECT AVG(x) AS xbar, STDDEV_POP(x) AS s, COUNT(*) AS n FROM base),
     nb    AS (SELECT b.h3, EXPLODE(h3_kring(b.h3, 1)) AS nh3 FROM base b),
     agg   AS (SELECT nb.h3, SUM(o.x) AS sx, COUNT(*) AS wi
               FROM nb JOIN base o ON nb.nh3 = o.h3 GROUP BY nb.h3)
SELECT agg.h3,
       (sx - xbar * wi) / (s * SQRT((n * wi - wi * wi) / (n - 1))) AS gi_spark
FROM agg CROSS JOIN stats
""").toPandas()
cmp = dg[["h3", "gi_z"]].merge(gi_spark, on="h3")
print(f"Correlación Gi* esda vs Spark SQL: {cmp[['gi_z','gi_spark']].corr().iloc[0,1]:.6f}")
print(f"Diferencia absoluta máxima: {(cmp.gi_z - cmp.gi_spark).abs().max():.2e}")